# Análisis de ventas de la tienda

Queremos identificar qué artículos se venden más y cómo cambian sus ventas según el día de la semana. La base se construyó a partir de fotografías de reportes operativos, cortes de caja, faltantes y movimientos por artículo.

El historial por artículo cubre tres mantecadas. Las comparaciones usan del 9 al 26 de octubre de 2026, su intervalo común.

## 1. Cargar la base SQLite

Ejecuta la celda y selecciona `ventas_tienda.sqlite`. Repite la carga cuando empieces una sesión nueva de Colab. Para que otra persona pueda reproducir el análisis, entrega el archivo SQLite junto al notebook.

In [1]:
from pathlib import Path
from google.colab import files

ruta_base = Path('ventas_tienda.sqlite')
if not ruta_base.is_file():
    archivos = files.upload()
    if ruta_base.name not in archivos:
        raise FileNotFoundError('Selecciona el archivo ventas_tienda.sqlite.')
    ruta_base.write_bytes(archivos[ruta_base.name])

print(f'Base disponible: {ruta_base.name}')

Saving ventas_tienda.sqlite to ventas_tienda.sqlite
Base disponible: ventas_tienda.sqlite


## 2. Verificar la conexión

SQLite ejecuta las consultas y pandas muestra los resultados como tablas. La conexión se abre en modo de solo lectura.

In [2]:
import sqlite3
import pandas as pd

ruta_sqlite = ruta_base.resolve().as_uri() + '?mode=ro'

def consultar(sql):
    conexion = sqlite3.connect(ruta_sqlite, uri=True)
    try:
        return pd.read_sql_query(sql, conexion)
    finally:
        conexion.close()

consultar('''
SELECT name AS objeto, type AS tipo
FROM sqlite_master
WHERE type IN ('table', 'view')
ORDER BY type, name
''')

,objeto,tipo
0,articulos,table
1,cobertura_articulos,table
2,cortes_caja,table
3,faltantes,table
4,fuentes,table
5,incidencias,table
6,indicador_huecos,table
7,indicador_venta_perdida,table
8,movimientos_articulo,table
9,resumen_diario,table


## 3. Ejecutar las consultas

Cada celda contiene una consulta de `consultas_iniciales.sql`. El texto entre comillas triples es SQL. La función `consultar()` lo ejecuta y devuelve una tabla.

### Ranking de los tres productos con datos en el mismo periodo

In [3]:
consultar("""
SELECT sku,
       descripcion,
       SUM(unidades_vendidas) AS unidades_vendidas,
       SUM(unidades_devueltas) AS unidades_devueltas,
       SUM(unidades_netas) AS unidades_netas
FROM ventas_producto_dia
WHERE fecha BETWEEN '2026-10-09' AND '2026-10-26'
GROUP BY sku, descripcion
ORDER BY unidades_netas DESC;
""")

,sku,descripcion,unidades_vendidas,unidades_devueltas,unidades_netas
0,7502294224733,Mantecada Bonamark chocolate 56 g,56,1,55
1,7502294224719,Mantecada Bonamark N 56 g,48,0,48
2,7502294224702,Mantecada Bonamark vainilla 56 g,31,0,31


### Unidades netas promedio por día de la semana y producto

In [4]:
consultar("""
WITH por_dia AS (
    SELECT sku, descripcion, dia_semana_0_domingo,
           COUNT(*) AS dias_observados,
           AVG(unidades_netas) AS promedio
    FROM ventas_producto_dia
    WHERE fecha BETWEEN '2026-10-09' AND '2026-10-26'
    GROUP BY sku, descripcion, dia_semana_0_domingo
)
SELECT sku, descripcion,
       CASE dia_semana_0_domingo
           WHEN 0 THEN 'Domingo'
           WHEN 1 THEN 'Lunes'
           WHEN 2 THEN 'Martes'
           WHEN 3 THEN 'Miércoles'
           WHEN 4 THEN 'Jueves'
           WHEN 5 THEN 'Viernes'
           WHEN 6 THEN 'Sábado'
       END AS dia_semana,
       dias_observados,
       ROUND(promedio, 2) AS unidades_netas_promedio
FROM por_dia
ORDER BY sku, promedio DESC, dia_semana_0_domingo;
""")

,sku,descripcion,dia_semana,dias_observados,unidades_netas_promedio
0,7502294224702,Mantecada Bonamark vainilla 56 g,Domingo,3,2.33
1,7502294224702,Mantecada Bonamark vainilla 56 g,Jueves,2,2.00
2,7502294224702,Mantecada Bonamark vainilla 56 g,Sábado,3,2.00
3,7502294224702,Mantecada Bonamark vainilla 56 g,Lunes,3,1.67
4,7502294224702,Mantecada Bonamark vainilla 56 g,Viernes,3,1.67
5,7502294224702,Mantecada Bonamark vainilla 56 g,Martes,2,1.50
6,7502294224702,Mantecada Bonamark vainilla 56 g,Miércoles,2,0.50
7,7502294224719,Mantecada Bonamark N 56 g,Jueves,2,5.50
8,7502294224719,Mantecada Bonamark N 56 g,Lunes,3,3.67
9,7502294224719,Mantecada Bonamark N 56 g,Miércoles,2,3.50


### Artículos más frecuentes en los listados de faltantes

In [5]:
consultar("""
SELECT f.sku, a.descripcion, COUNT(DISTINCT f.fecha) AS dias_reportados
FROM faltantes AS f
JOIN articulos AS a ON a.sku = f.sku
GROUP BY f.sku, a.descripcion
ORDER BY dias_reportados DESC, a.descripcion;
""")

,sku,descripcion,dias_reportados
0,7500465392410,Bloom Her desodorante antitranspirante,11
1,7622210576200,Chips Ahoy regular 128.8 g,11
2,8699141055035,Galleta cubierta con chocolate,11
3,7502294221442,Galleta wafer napolitana chocolate,11
4,7500435019828,HS Suave M 375 ml,11
...,...,...,...
103,041789001925,Sopa Maruchan carne,1
104,7501054674715,Suavitel Morning Sun 700 ml,1
105,7502294225402,Sándwich chocolate Bonamark 101 g,1
106,7501036524121,Toallas de papel Regio Rinde+,1


### Ventas y clientes del resumen operativo, con su foto de origen

In [6]:
consultar("""
SELECT r.fecha, r.hora, r.venta_sin_iva_sin_tae_mxn, r.trafico_clientes,
       r.ticket_promedio_reportado_mxn, f.archivo AS foto
FROM resumen_diario AS r
JOIN fuentes AS f ON f.id = r.fuente_id
ORDER BY r.fecha;
""")

,fecha,hora,venta_sin_iva_sin_tae_mxn,trafico_clientes,ticket_promedio_reportado_mxn,foto
0,2026-10-31,21:59:45,75459.54,590,129.01,WhatsApp Image 2026-10-31 at 22.03.10.jpeg
1,2026-11-06,21:35:08,46190.19,500,93.57,WhatsApp Image 2026-11-06 at 21.35.49.jpeg
2,2026-11-07,21:53:48,56505.78,521,109.53,WhatsApp Image 2026-11-07 at 21.54.56.jpeg
3,2026-11-08,21:36:48,85287.84,620,137.70,WhatsApp Image 2026-11-08 at 21.38.40 (1).jpeg
4,2026-11-09,21:39:34,60663.63,527,115.50,WhatsApp Image 2026-11-09 at 21.40.55.jpeg
5,2026-11-10,21:36:32,56326.18,544,104.70,WhatsApp Image 2026-11-10 at 21.38.32.jpeg
6,2026-11-11,21:37:21,43752.14,504,88.16,WhatsApp Image 2026-11-11 at 21.38.21.jpeg
7,2026-11-12,21:39:10,44025.86,464,96.24,WhatsApp Image 2026-11-12 at 21.40.48.jpeg
8,2026-11-13,22:18:49,52032.52,496,105.21,WhatsApp Image 2026-11-13 at 22.20.40.jpeg
9,2026-11-14,21:50:21,62739.37,521,121.20,WhatsApp Image 2026-11-14 at 21.51.18.jpeg


### Fechas sin resumen o sin corte, dentro del intervalo del ZIP

In [7]:
consultar("""
SELECT fecha, tiene_resumen, cortes_fotografiados, faltantes_transcritos
FROM cobertura_diaria
WHERE tiene_resumen = 0 OR cortes_fotografiados = 0
ORDER BY fecha;
""")

,fecha,tiene_resumen,cortes_fotografiados,faltantes_transcritos
0,2026-10-28,0,0,0
1,2026-10-29,0,0,0
2,2026-10-30,0,0,0
3,2026-11-01,0,0,0
4,2026-11-02,0,0,0
5,2026-11-03,0,0,0
6,2026-11-04,0,0,0
7,2026-11-05,0,0,0
8,2026-11-07,1,0,37


### Incidencias de captura y datos pendientes de confirmar

In [8]:
consultar("""
SELECT i.tema, i.detalle, f.archivo AS foto
FROM incidencias AS i
LEFT JOIN fuentes AS f ON f.id = i.fuente_id
ORDER BY i.id;
""")

,tema,detalle,foto
0,Duplicado exacto,Es el mismo archivo que la fuente 2 según SHA-...,WhatsApp Image 2026-11-03 at 07.16.33.jpeg
1,Listado repetido,Dos copias del listado del 7 de noviembre en l...,WhatsApp Image 2026-11-07 at 21.54.56.jpeg
2,Hora ilegible,La hora de cierre del corte 4261 está recortad...,WhatsApp Image 2026-11-10 at 21.38.32 (1).jpeg
3,SKU sin confirmar,Hay bloques de movimientos con el encabezado t...,WhatsApp Image 2026-10-30 at 15.33.11.jpeg
4,SKU sin confirmar,Se incorporaron vainilla y chocolate al concil...,WhatsApp Image 2026-10-31 at 22.04.54.jpeg
5,Fecha impresa inconsistente,El folio 2467 por 6 unidades del SKU 750229422...,WhatsApp Image 2026-10-28 at 15.45.28.jpeg
6,Cantidad actualizada,"Para chocolate, el folio 2473 pasa de 1 a 2 un...",WhatsApp Image 2026-10-29 at 15.47.51.jpeg
7,Venta perdida por artículo parcial,Se transcribieron los primeros diez renglones ...,WhatsApp Image 2026-11-14 at 21.51.18.jpeg
8,Venta perdida por artículo parcial,Se transcribieron los primeros diez renglones ...,WhatsApp Image 2026-11-15 at 21.50.06.jpeg
9,Corte parcial visible,Hay un reporte adicional cortado en el borde i...,WhatsApp Image 2026-11-06 at 21.35.49 (1).jpeg


## Interpretación

En el periodo común, la mantecada de chocolate registró 55 unidades netas, seguida de la mantecada N con 48 y la de vainilla con 31. Esta comparación cubre los tres productos con historial en la base. Cada día de la semana tiene dos o tres fechas observadas, por lo que los promedios son descriptivos de la muestra.

Los listados de faltantes y los indicadores de venta perdida se pueden analizar por separado. Los cortes por caja pueden abarcar parte del día anterior. Para consultar ventas de tienda por fecha, usa el resumen operativo.